<a href="https://colab.research.google.com/github/barr92/public-house-price-data/blob/main/EPC_Pipeline_for_Square_Feet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Signals BI — EPC Pipeline
**Project:** `signalsbi-new` | **Dataset:** `epc` (prod) / `epc_dev` (dev)

## How to use
| Scenario | Settings |
|---|---|
| First ever run | `ENV = 'dev'`, `FIRST_RUN = True` |
| Monthly update | `ENV = 'prod'`, `FIRST_RUN = False` |
| Promote dev → prod | Run Cell 8 with `PROMOTE = True` |

## Tables produced
| Table | Description |
|---|---|
| `epc_raw` | Every certificate from the bulk download, append-only |
| `epc_clean` | Latest certificate per UPRN, deduplicated |
| `epc_transaction_matches` | EPC matched to `transactions_raw` at property level |
| `sector_price_per_sqft` | Aggregated price per sq ft per sector × property type |

## 0. Authenticate

In [1]:
from google.colab import auth
auth.authenticate_user()
print('Authenticated')

Authenticated


## 1. Install & import

In [ ]:
# !pip install google-cloud-bigquery google-cloud-bigquery-storage google-cloud-storage pandas db-dtypes requests --quiet

import io
import zipfile
import requests
import pandas as pd
from datetime import datetime, timezone
from google.cloud import bigquery, storage

print('Imports OK')

## 2. Configuration
**Only cell you need to change between runs.**

In [ ]:
# ── CHANGE THESE BETWEEN RUNS ────────────────────────────────────────────────
ENV       = 'dev'   # 'dev' or 'prod'
FIRST_RUN = True    # True = full bulk download | False = monthly update only
# ─────────────────────────────────────────────────────────────────────────────

PROJECT   = 'signalsbi-new'
DATASETS  = {'prod': 'epc', 'dev': 'epc_dev'}
DATASET   = DATASETS[ENV]
D         = f'`{PROJECT}.{DATASET}`'
GCS_BUCKET  = 'signals-new'
GCS_PREFIX  = 'property-details/epc/'

# EPC API
EPC_API_BASE     = 'https://api.get-energy-performance-data.communities.gov.uk'
EPC_BEARER_TOKEN = 'YOUR_BEARER_TOKEN_HERE'  # replace with your token
EPC_HEADERS      = {
    'Authorization': f'Bearer {EPC_BEARER_TOKEN}',
    'Accept': 'application/json',
}

# BigQuery clients
bq  = bigquery.Client(project=PROJECT, location='EU')
gcs = storage.Client(project=PROJECT)

print(f'Environment : {ENV.upper()}')
print(f'First run   : {FIRST_RUN}')
print(f'Dataset     : {PROJECT}.{DATASET}')

## 3. Create dataset and tables

In [ ]:
# Create dataset
ds = bigquery.Dataset(f'{PROJECT}.{DATASET}')
ds.location = 'EU'
bq.create_dataset(ds, exists_ok=True)
print(f'Dataset `{PROJECT}.{DATASET}` ready.')

ddls = [

# 1. epc_raw — append-only, every certificate from every download
f'''
CREATE TABLE IF NOT EXISTS {D}.epc_raw (
  lmk_key                     STRING,
  uprn                        STRING,
  uprn_source                 STRING,
  address1                    STRING,
  address2                    STRING,
  address3                    STRING,
  postcode                    STRING,
  postcode_sector             STRING,
  postcode_district           STRING,
  local_authority             STRING,
  constituency                STRING,
  built_form                  STRING,
  property_type               STRING,
  total_floor_area_sqm        FLOAT64,
  total_floor_area_sqft       FLOAT64,
  number_habitable_rooms      INT64,
  number_heated_rooms         INT64,
  current_energy_rating       STRING,
  current_energy_efficiency   INT64,
  potential_energy_rating     STRING,
  potential_energy_efficiency INT64,
  construction_age_band       STRING,
  lodgement_date              DATE,
  lodgement_datetime          TIMESTAMP,
  inspection_date             DATE,
  transaction_type            STRING,
  tenure                      STRING,
  mains_gas_flag              STRING,
  floor_level                 STRING,
  flat_top_storey             STRING,
  ingested_at                 TIMESTAMP,
  file_source                 STRING
)
PARTITION BY lodgement_date
CLUSTER BY postcode_sector, property_type, uprn
OPTIONS (require_partition_filter = false)
''',

# 2. epc_clean — latest cert per UPRN (CTAS — not pre-created)
# Created by Cell 5

# 3. epc_transaction_matches — EPC matched to transactions_raw
# Created by Cell 6

# 4. sector_price_per_sqft — aggregated
# Created by Cell 7
]

for ddl in ddls:
    bq.query(ddl).result()

print('Tables ready.')

## 4. Download EPC data and load into BigQuery
- **FIRST_RUN = True** → calls bulk download API, streams zip to GCS, loads all CSVs into `epc_raw`
- **FIRST_RUN = False** → loads only the latest monthly file added to GCS

In [ ]:
# EPC CSV column mapping — new portal column names → our schema
COL_MAP = {
    'LMK_KEY':                      'lmk_key',
    'UPRN':                         'uprn',
    'UPRN_SOURCE':                  'uprn_source',
    'ADDRESS1':                     'address1',
    'ADDRESS2':                     'address2',
    'ADDRESS3':                     'address3',
    'POSTCODE':                     'postcode',
    'LOCAL_AUTHORITY_LABEL':        'local_authority',
    'CONSTITUENCY_LABEL':           'constituency',
    'BUILT_FORM':                   'built_form',
    'PROPERTY_TYPE':                'property_type',
    'TOTAL_FLOOR_AREA':             'total_floor_area_sqm',
    'NUMBER_HABITABLE_ROOMS':       'number_habitable_rooms',
    'NUMBER_HEATED_ROOMS':          'number_heated_rooms',
    'CURRENT_ENERGY_RATING':        'current_energy_rating',
    'CURRENT_ENERGY_EFFICIENCY':    'current_energy_efficiency',
    'POTENTIAL_ENERGY_RATING':      'potential_energy_rating',
    'POTENTIAL_ENERGY_EFFICIENCY':  'potential_energy_efficiency',
    'CONSTRUCTION_AGE_BAND':        'construction_age_band',
    'LODGEMENT_DATE':               'lodgement_date',
    'LODGEMENT_DATETIME':           'lodgement_datetime',
    'INSPECTION_DATE':              'inspection_date',
    'TRANSACTION_TYPE':             'transaction_type',
    'TENURE':                       'tenure',
    'MAINS_GAS_FLAG':               'mains_gas_flag',
    'FLOOR_LEVEL':                  'floor_level',
    'FLAT_TOP_STOREY':              'flat_top_storey',
}

def parse_epc_csv(raw_bytes, file_source):
    df = pd.read_csv(io.BytesIO(raw_bytes), dtype=str, low_memory=False)
    # Rename columns that exist
    df = df.rename(columns={k: v for k, v in COL_MAP.items() if k in df.columns})
    # Keep only mapped columns
    keep = [v for v in COL_MAP.values() if v in df.columns]
    df = df[keep].copy()

    # Type casting
    df['total_floor_area_sqm']      = pd.to_numeric(df.get('total_floor_area_sqm'), errors='coerce')
    df['total_floor_area_sqft']     = (df['total_floor_area_sqm'] * 10.7639).round(1)
    df['current_energy_efficiency'] = pd.to_numeric(df.get('current_energy_efficiency'), errors='coerce').astype('Int64')
    df['potential_energy_efficiency']= pd.to_numeric(df.get('potential_energy_efficiency'), errors='coerce').astype('Int64')
    df['number_habitable_rooms']    = pd.to_numeric(df.get('number_habitable_rooms'), errors='coerce').astype('Int64')
    df['number_heated_rooms']       = pd.to_numeric(df.get('number_heated_rooms'), errors='coerce').astype('Int64')
    df['lodgement_date']            = pd.to_datetime(df.get('lodgement_date'), errors='coerce').dt.date
    df['inspection_date']           = pd.to_datetime(df.get('inspection_date'), errors='coerce').dt.date
    df['lodgement_datetime']        = pd.to_datetime(df.get('lodgement_datetime'), errors='coerce', utc=True)

    # Derive postcode sector and district
    df['postcode']          = df['postcode'].str.strip().str.upper()
    df['postcode_sector']   = df['postcode'].str.rsplit(' ', n=1).str[0] + ' ' + df['postcode'].str[-3]
    df['postcode_district'] = df['postcode'].str.split(' ').str[0]

    # Drop rows with no UPRN and no floor area
    df = df.dropna(subset=['lmk_key'])
    df = df[df['total_floor_area_sqm'] > 0]

    df['ingested_at']  = datetime.now(timezone.utc)
    df['file_source']  = file_source

    return df

def load_to_bq(df, label):
    print(f'  Loading {len(df):,} rows ({label}) to epc_raw...', end=' ', flush=True)
    bq.load_table_from_dataframe(
        df, f'{PROJECT}.{DATASET}.epc_raw',
        job_config=bigquery.LoadJobConfig(write_disposition='WRITE_APPEND')
    ).result()
    print('done')

def stream_zip_to_bq(zip_bytes, source_label):
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
        csv_files = [f for f in z.namelist() if f.endswith('.csv')]
        print(f'  Found {len(csv_files)} CSV files in zip')
        for fname in sorted(csv_files):
            print(f'  Processing {fname}...', end=' ', flush=True)
            with z.open(fname) as f:
                raw = f.read()
            df = parse_epc_csv(raw, f'{source_label}/{fname}')
            print(f'{len(df):,} rows')
            load_to_bq(df, fname)

# ── Main ingest ───────────────────────────────────────────────────────────
if FIRST_RUN:
    print('FIRST RUN — downloading full EPC dataset via API...')
    # Step 1: Get signed S3 URL via 302 redirect
    resp = requests.get(
        f'{EPC_API_BASE}/api/files/domestic/csv',
        headers=EPC_HEADERS,
        allow_redirects=False,
        timeout=30
    )
    if resp.status_code != 302:
        raise RuntimeError(f'Expected 302 redirect, got {resp.status_code}: {resp.text}')

    download_url = resp.headers['location']
    print(f'  Redirect URL obtained. Downloading...')

    # Step 2: Download the zip from S3 (no auth needed for signed URL)
    dl = requests.get(download_url, timeout=600, stream=True)
    dl.raise_for_status()
    chunks, mb = [], 0
    for chunk in dl.iter_content(1024 * 1024):
        chunks.append(chunk)
        mb += len(chunk) / 1024 / 1024
        print(f'  {mb:.0f} MB downloaded...', end='\r', flush=True)
    zip_bytes = b''.join(chunks)
    print(f'  Download complete: {mb:.0f} MB')

    # Step 3: Also save to GCS for backup
    print('  Saving to GCS...')
    bucket = gcs.bucket(GCS_BUCKET)
    blob = bucket.blob(f'{GCS_PREFIX}domestic-full-{datetime.now().strftime("%Y-%m")}.zip')
    blob.upload_from_string(zip_bytes, content_type='application/zip')
    print('  GCS backup saved.')

    # Step 4: Process zip into BigQuery
    stream_zip_to_bq(zip_bytes, 'full-load')

else:
    print('MONTHLY UPDATE — loading latest file from GCS...')
    # List GCS files and pick the most recent
    bucket = gcs.bucket(GCS_BUCKET)
    blobs  = sorted(
        [b for b in bucket.list_blobs(prefix=GCS_PREFIX) if b.name.endswith('.zip')],
        key=lambda b: b.updated, reverse=True
    )
    if not blobs:
        raise RuntimeError('No EPC zip files found in GCS. Run FIRST_RUN = True first.')

    latest_blob = blobs[0]
    print(f'  Latest file: {latest_blob.name} (updated {latest_blob.updated})')
    zip_bytes = latest_blob.download_as_bytes()
    stream_zip_to_bq(zip_bytes, latest_blob.name)

n = bq.query(f'SELECT COUNT(*) AS n FROM {D}.epc_raw').to_dataframe().iloc[0]['n']
print(f'\nepc_raw total rows: {n:,}')

## 5. Build `epc_clean`
One row per UPRN — latest certificate only.

In [ ]:
print('Building epc_clean...')
bq.query(f'''
CREATE OR REPLACE TABLE {D}.epc_clean AS

WITH ranked AS (
  SELECT *,
    ROW_NUMBER() OVER (
      PARTITION BY uprn
      ORDER BY lodgement_date DESC, ingested_at DESC
    ) AS rn
  FROM {D}.epc_raw
  WHERE uprn IS NOT NULL AND uprn != ''
    AND total_floor_area_sqm > 0
)
SELECT
  lmk_key, uprn, uprn_source,
  address1, address2, address3,
  postcode, postcode_sector, postcode_district,
  local_authority, constituency,
  built_form, property_type,
  total_floor_area_sqm, total_floor_area_sqft,
  number_habitable_rooms, number_heated_rooms,
  current_energy_rating, current_energy_efficiency,
  potential_energy_rating, potential_energy_efficiency,
  construction_age_band,
  lodgement_date, inspection_date,
  transaction_type, tenure,
  mains_gas_flag, floor_level, flat_top_storey,
  ingested_at
FROM ranked
WHERE rn = 1
''').result()

n = bq.query(f'SELECT COUNT(*) AS n FROM {D}.epc_clean').to_dataframe().iloc[0]['n']
print(f'epc_clean: {n:,} unique properties')

## 6. Match EPC to transactions
Matches EPC floor area to `transactions_raw` using a tiered approach:
- **HIGH confidence**: UPRN match (if UPRN exists in both)
- **MEDIUM confidence**: PAON + street + postcode match
- **LOW confidence**: postcode + property type match (detached only)

Only transactions from 2012 onwards are matched (EPC coverage before 2012 is sparse).

In [ ]:
print('Building epc_transaction_matches...')
bq.query(f'''
CREATE OR REPLACE TABLE {D}.epc_transaction_matches AS

WITH transactions AS (
  SELECT
    transaction_id, transaction_date, price,
    postcode, postcode_sector, postcode_district,
    property_type, tenure, new_build,
    paon, saon, street, town, local_authority,
    year_month, year
  FROM `signalsbi-new.uk_sold_prices.transactions_raw`
  WHERE record_status != 'D'
    AND year >= 2012
    AND price > 10000
),

-- Tier 1: PAON + street + postcode match
tier1 AS (
  SELECT
    t.transaction_id,
    e.uprn,
    e.lmk_key,
    e.total_floor_area_sqm,
    e.total_floor_area_sqft,
    e.current_energy_rating,
    e.current_energy_efficiency,
    e.construction_age_band,
    e.built_form,
    e.lodgement_date,
    'PAON_STREET_POSTCODE' AS match_method,
    'MEDIUM'               AS match_confidence
  FROM transactions t
  JOIN `{D}`.epc_clean e
    ON UPPER(TRIM(t.postcode)) = UPPER(TRIM(e.postcode))
    AND UPPER(TRIM(t.paon))   = UPPER(TRIM(e.address1))
    AND UPPER(TRIM(t.street)) = UPPER(TRIM(e.address2))
  WHERE t.paon IS NOT NULL AND t.street IS NOT NULL
),

-- Tier 2: Postcode only for Detached (usually one per postcode)
tier2 AS (
  SELECT
    t.transaction_id,
    e.uprn,
    e.lmk_key,
    e.total_floor_area_sqm,
    e.total_floor_area_sqft,
    e.current_energy_rating,
    e.current_energy_efficiency,
    e.construction_age_band,
    e.built_form,
    e.lodgement_date,
    'POSTCODE_DETACHED' AS match_method,
    'LOW'               AS match_confidence
  FROM transactions t
  JOIN `{D}`.epc_clean e
    ON UPPER(TRIM(t.postcode)) = UPPER(TRIM(e.postcode))
    AND e.built_form = 'Detached'
  WHERE t.property_type = 'Detached'
    AND t.transaction_id NOT IN (SELECT transaction_id FROM tier1)
  QUALIFY ROW_NUMBER() OVER (PARTITION BY t.transaction_id ORDER BY e.lodgement_date DESC) = 1
),

combined AS (
  SELECT * FROM tier1
  UNION ALL
  SELECT * FROM tier2
)

SELECT
  t.transaction_id, t.transaction_date, t.price,
  t.postcode, t.postcode_sector, t.postcode_district,
  t.property_type, t.tenure, t.new_build,
  t.paon, t.street, t.town, t.local_authority,
  t.year_month, t.year,
  c.uprn, c.lmk_key,
  c.total_floor_area_sqm, c.total_floor_area_sqft,
  c.current_energy_rating, c.current_energy_efficiency,
  c.construction_age_band, c.built_form,
  c.lodgement_date AS epc_lodgement_date,
  c.match_method, c.match_confidence,
  ROUND(SAFE_DIVIDE(t.price, c.total_floor_area_sqft), 0) AS price_per_sqft,
  ROUND(SAFE_DIVIDE(t.price, c.total_floor_area_sqm), 0)  AS price_per_sqm,
  CURRENT_TIMESTAMP() AS refreshed_at
FROM transactions t
JOIN combined c USING (transaction_id)
''').result()

n = bq.query(f'SELECT COUNT(*) AS n FROM {D}.epc_transaction_matches').to_dataframe().iloc[0]['n']
print(f'epc_transaction_matches: {n:,} matched transactions')

# Match rate check
total = bq.query(
    'SELECT COUNT(*) AS n FROM `signalsbi-new.uk_sold_prices.transactions_raw` '
    'WHERE year >= 2012 AND price > 10000 AND record_status != \'D\''
).to_dataframe().iloc[0]['n']
print(f'Match rate: {n/total*100:.1f}% of post-2012 transactions matched')

## 7. Build `sector_price_per_sqft`

In [ ]:
print('Building sector_price_per_sqft...')
bq.query(f'''
CREATE OR REPLACE TABLE {D}.sector_price_per_sqft AS

WITH base AS (
  SELECT *
  FROM {D}.epc_transaction_matches
  WHERE price_per_sqft BETWEEN 10 AND 5000  -- exclude extreme outliers
    AND total_floor_area_sqft > 100           -- exclude implausibly small
    AND total_floor_area_sqft < 10000         -- exclude implausibly large
),
all_type AS (
  SELECT postcode_sector, postcode_district, property_type, year,
         price_per_sqft, price_per_sqm, total_floor_area_sqft, total_floor_area_sqm,
         match_confidence
  FROM base
  UNION ALL
  SELECT postcode_sector, postcode_district, 'All' AS property_type, year,
         price_per_sqft, price_per_sqm, total_floor_area_sqft, total_floor_area_sqm,
         match_confidence
  FROM base
)
SELECT
  postcode_sector,
  postcode_district,
  property_type,
  year,
  COUNT(*)                                                     AS transaction_count,
  CAST(ROUND(AVG(price_per_sqft), 0) AS INT64)                 AS avg_price_per_sqft,
  CAST(APPROX_QUANTILES(price_per_sqft, 100)[OFFSET(50)] AS INT64) AS median_price_per_sqft,
  CAST(ROUND(AVG(price_per_sqm), 0) AS INT64)                  AS avg_price_per_sqm,
  CAST(ROUND(AVG(total_floor_area_sqft), 0) AS INT64)          AS avg_floor_area_sqft,
  CAST(ROUND(AVG(total_floor_area_sqm), 0) AS INT64)           AS avg_floor_area_sqm,
  COUNTIF(match_confidence = 'MEDIUM')                         AS high_confidence_count,
  COUNTIF(match_confidence = 'LOW')                            AS low_confidence_count,
  ROUND(COUNTIF(match_confidence = 'MEDIUM') / COUNT(*) * 100, 1) AS medium_confidence_pct,
  CURRENT_TIMESTAMP()                                          AS refreshed_at
FROM all_type
GROUP BY 1, 2, 3, 4
ORDER BY 1, 3, 4
''').result()

n = bq.query(f'SELECT COUNT(*) AS n FROM {D}.sector_price_per_sqft').to_dataframe().iloc[0]['n']
print(f'sector_price_per_sqft: {n:,} rows')

## 8. Pipeline summary

In [ ]:
tables = [
    ('epc_raw',                  'ingested_at'),
    ('epc_clean',                'ingested_at'),
    ('epc_transaction_matches',  'refreshed_at'),
    ('sector_price_per_sqft',    'refreshed_at'),
]
parts = ' UNION ALL '.join([
    f"SELECT '{t}' AS table_name, COUNT(*) AS row_count, MAX(CAST({ts} AS STRING)) AS last_updated FROM {D}.{t}"
    for t, ts in tables
])
summary = bq.query(parts + ' ORDER BY table_name').to_dataframe()
print(f'=== EPC Pipeline [{ENV.upper()}] ===')
print(f'Dataset : {PROJECT}.{DATASET}\n')
print(summary.to_string(index=False))

## 9. Promote dev → prod
Run manually after validating dev. Set `PROMOTE = True` to execute.

In [ ]:
PROMOTE = False  # <-- set True to copy dev -> prod

if not PROMOTE:
    print('Promotion skipped. Set PROMOTE = True to copy dev -> prod.')
else:
    if ENV != 'dev':
        raise RuntimeError("Set ENV = 'dev' before promoting.")
    prod_dataset = DATASETS['prod']
    to_promote = [t for t, _ in tables]
    for table in to_promote:
        src = f'{PROJECT}.{DATASETS["dev"]}.{table}'
        dst = f'{PROJECT}.{prod_dataset}.{table}'
        print(f'  Copying {table}...', end=' ', flush=True)
        try:
            bq.copy_table(src, dst, job_config=bigquery.CopyJobConfig(
                write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE)).result()
            print('done')
        except Exception as e:
            print(f'SKIPPED ({e})')
    print(f'Promotion complete -> {PROJECT}.{prod_dataset}')